# 🧠 Python Frameworks Assignment: Professional Workflow (Colab + Kaggle API)

## 🔧 Tools Required

- Python 3.7+

- pandas

- matplotlib / seaborn

- Streamlit (for local or cloud deployment)

- GitHub (for submission)

- Google Colab (for development)

## ⚙️ Step 0: Setup Kaggle API in Colab

### 🔹 1. Get Kaggle API credentials

- Go to [Kaggle Account Settings](https://www.kaggle.com/account)

- Scroll to API section and click Create New API Token

- This downloads a file called kaggle.json

### 🔹 2. Upload kaggle.json to Colab

In [59]:
from google.colab import files
files.upload()  # Upload kaggle.json here


Saving kaggle.json to kaggle (3).json


{'kaggle (3).json': b'{"username":"frederick001","key":"4d66bbb7b0cf5104d25b605d2dc98797"}'}

### 🔹 3. Configure Kaggle API

In [60]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json


### 🔹 4. Download only the metadata file

In [65]:
import json

kaggle_token = {
    "username":"frederick001",
    "key":"4d66bbb7b0cf5104d25b605d2dc98797"
    }

with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump(kaggle_token, f)

!chmod 600 /root/.kaggle/kaggle.json

!kaggle datasets download -d allen-institute-for-ai/CORD-19-research-challenge --file metadata.csv
!unzip metadata.csv.zip


Dataset URL: https://www.kaggle.com/datasets/allen-institute-for-ai/CORD-19-research-challenge
License(s): other
 94% 526M/560M [00:04<00:00, 148MB/s]
100% 560M/560M [00:07<00:00, 75.0MB/s]
unzip:  cannot find or open metadata.csv.zip, metadata.csv.zip.zip or metadata.csv.zip.ZIP.


## 📥 Part 1: Data Loading & Basic Exploration

In [67]:
import pandas as pd

# Read the CSV file
# Use encoding='latin-1', header=0, on_bad_lines='skip', and engine='python' for correct parsing
# Remove low_memory=False as it is not compatible with engine='python'
df = pd.read_csv('metadata.csv', encoding='latin-1', header=0, on_bad_lines='skip', engine='python')

# Quick checks
print(df.shape)
print(df.info())
print(df.head())

(4355251, 3)
<class 'pandas.core.frame.DataFrame'>
Index: 4355251 entries, ÝnË"SÚËN¬ to ½vN^§ÁÛ±³
Data columns (total 3 columns):
 #   Column                                                                                                                                                                                                                                                                                                                  Dtype 
---  ------                                                                                                                                                                                                                                                                                                                  ----- 
 0   PK-    ¤ÆTäg¶Kÿÿÿÿÿÿÿÿ  metadata.csv  tÜHb    ~#    ¼½KÜÈ&¸o ÿ!à"#ÐôI§¦§z+S!E+u«Q¸0æîTÐIO>"äZ%0ZOÐ]Ëô¦g5Ý³jÕ¹ùùKæ|çIÒSUÂ½uîN£Ù±cç}>KÊ*ýK¥~½Q~]¶U¢ÿòÑo²&×~ZfþnÐ»6Þêô/ô¯<KtQk_Åu

### 🔹 Basic Exploration

In [68]:
# Missing values
print(df.isnull().sum())

# Summary stats
print(df.describe(include='all'))


PK-    ¤ÆTäg¶Kÿÿÿÿÿÿÿÿ  metadata.csv  tÜHb    ~#    ¼½KÜÈ&¸o ÿ!à"#ÐôI§¦§z+S!E+u«Q¸0æîTÐIO>"äZ%0ZOÐ]Ëô¦g5Ý³jÕ¹ùùKæ|çIÒSUÂ½uîN£Ù±cç}>KÊ*ýK¥~½Q~]¶U¢ÿòÑo²&×~ZfþnÐ»6Þêô/ô¯<KtQk_ÅuS©¤ÁWyVoþÒd[ú´m6eUûh BåþV­ñÐí¦üKRÞd©.hxú@U³üc®þò¡.¿¬²\×x[ÿÏ¶Êý:¢ßýÛÓ®ç7åòJ­ây³Ù"¢ùb¶Z    2944696
£å*]&Ë X                                                                                                                                                                                                                                                                                                                  3917683
ç«Ø¿¼xâ?xgE¨Ü[iÕ´®½rå%mvUy£ïb»\Õ[åí                                                                                                                                                                                                                                                                              4244735
dtype: int64
       PK-    ¤ÆT

## 🧹 Part 2: Data Cleaning & Preparation

### 🔹 Handle Missing Data

In [70]:
# Fill abstract with empty string *before* dropping columns based on threshold
# This ensures the 'abstract' column is not dropped due to missing values
if 'abstract' in df.columns:
    df['abstract'] = df['abstract'].fillna('')
    print("Missing values in 'abstract' filled with empty string.")
else:
    print("Warning: 'abstract' column not found in DataFrame.")

# Drop columns with >50% missing values in the original full DataFrame
# Use the threshold calculated based on the full dataset length
# Note: This threshold might be less relevant for the sample data loaded in df
threshold = len(df) * 0.5 # Recalculate threshold based on the loaded sample size
df_processed = df.dropna(thresh=threshold, axis=1)

# Quick check after processing
print("\nShape after dropping columns:", df_processed.shape)
print("\nMissing values after dropping columns:")
print(df_processed.isnull().sum())


Shape after dropping columns: (4355251, 0)

Missing values after dropping columns:
Series([], dtype: float64)


### 🔹 Prepare for Analysis

In [73]:
# Start with the processed DataFrame from the previous step
df_cleaned = df_processed.copy()

# Convert publish_time to datetime
if 'publish_time' in df_cleaned.columns:
    df_cleaned['publish_time'] = pd.to_datetime(df_cleaned['publish_time'], errors='coerce')
    # Extract year
    df_cleaned['year'] = df_cleaned['publish_time'].dt.year
else:
    print("Warning: 'publish_time' column not found in df_cleaned. Cannot convert to datetime or extract year.")
    # df_cleaned['year'] = None # Setting year to None might cause issues later, better to not create the column if publish_time is missing

# Abstract word count
if 'abstract' in df_cleaned.columns:
    df_cleaned['abstract_word_count'] = df_cleaned['abstract'].apply(lambda x: len(str(x).split()))
else:
    print("Warning: 'abstract' column not found in df_cleaned. Cannot calculate abstract word count.")
    # df_cleaned['abstract_word_count'] = 0 # Setting to 0 might be misleading, better to not create if abstract is missing


# Quick check after preparation - select only existing columns for info and isnull().sum()
cols_to_check = [col for col in ['publish_time', 'year', 'abstract_word_count'] if col in df_cleaned.columns]

if cols_to_check:
    display(df_cleaned[cols_to_check].head())
    print(df_cleaned[cols_to_check].info())
    print("\nMissing values after preparation:")
    print(df_cleaned[cols_to_check].isnull().sum())
else:
    print("\nNo relevant columns found for checking after preparation.")


No relevant columns found for checking after preparation.


### 📊 Part 3: Analysis & Visualization

### 🔹 Publications by Year

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

year_counts = df_cleaned['year'].value_counts().sort_index()
sns.barplot(x=year_counts.index, y=year_counts.values)
plt.title('Publications by Year')
plt.xlabel('Year')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.show()


### 🔹 Top Journals

In [ ]:
top_journals = df_cleaned['journal'].value_counts().head(10)
top_journals.plot(kind='barh', title='Top Journals Publishing COVID-19 Research')
plt.xlabel('Number of Papers')
plt.show()


### 🔹 Word Frequency in Titles

In [ ]:
from collections import Counter
import re

titles = df_cleaned['title'].dropna().str.lower().str.replace(r'[^\w\s]', '', regex=True)
word_counts = Counter(" ".join(titles).split())
common_words = dict(word_counts.most_common(20))

plt.bar(common_words.keys(), common_words.values())
plt.xticks(rotation=45)
plt.title('Common Words in Titles')
plt.show()


### 🔹 Word Cloud

In [ ]:
from wordcloud import WordCloud

text = " ".join(df_cleaned['title'].dropna())
wordcloud = WordCloud(width=800, height=400).generate(text)

plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud of Titles')
plt.show()


## 🌐 Part 4: Streamlit App (Run Locally or Deploy)

### 🔹 Sample app.py

In [ ]:
import streamlit as st
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('metadata.csv', low_memory=False)
df['publish_time'] = pd.to_datetime(df['publish_time'], errors='coerce')
df['year'] = df['publish_time'].dt.year

st.title("CORD-19 Data Explorer")
st.write("Explore COVID-19 research metadata")

year_range = st.slider("Select year range", 2019, 2022, (2020, 2021))
filtered = df[df['year'].between(year_range[0], year_range[1])]

st.write(filtered[['title', 'journal', 'year']].head())

year_counts = filtered['year'].value_counts().sort_index()
st.bar_chart(year_counts)


✅ Run locally: streamlit run app.py

✅ Or deploy via [Streamlit Cloud](https://streamlit.io/cloud)